yt-dlp -x --audio-format mp3 --postprocessor-args "-ac 1 -ar 16000 -b:a 64k" --output "D:/WorkD/pythonProjs/ml_algo/lection_conspect/data_raw/%(playlist_index)04d_%(id)s_%(title).50s.%(ext)s" --restrict-filenames --no-overwrites --download-archive downloaded.txt --sleep-requests 1 --sleep-interval 5 --max-sleep-interval 10 --throttled-rate 50K --match-filter "duration >= 3600 and duration < 7200" --max-downloads 700 --embed-thumbnail --no-check-certificate https://www.youtube.com/@lecturesMEPhI

yt-dlp -x --audio-format mp3 --postprocessor-args "-ac 1 -ar 16000 -b:a 64k" --output "D:/WorkD/pythonProjs/ml_algo/lection_conspect/data_raw/%(playlist_index)04d_%(id)s_%(title).50s.%(ext)s" --restrict-filenames --no-overwrites --download-archive downloaded.txt --sleep-requests 1 --sleep-interval 5 --max-sleep-interval 10 --throttled-rate 50K --match-filter "duration >= 3600 and duration < 7200" --max-downloads 700 --embed-thumbnail --no-check-certificate https://www.youtube.com/@mathematicsathse1021

yt-dlp -x --audio-format mp3 --postprocessor-args "-ac 1 -ar 16000 -b:a 64k" --output "D:/WorkD/pythonProjs/ml_algo/lection_conspect/data_raw/%(playlist_index)04d_%(id)s_%(title).50s.%(ext)s" --restrict-filenames --no-overwrites --download-archive downloaded.txt --sleep-requests 1 --sleep-interval 5 --max-sleep-interval 10 --throttled-rate 50K --match-filter "duration >= 3600 and duration < 7200" --max-downloads 700 --embed-thumbnail --no-check-certificate https://www.youtube.com/@lectory_fpmi

In [1]:
import torch, numpy as np, librosa, pathlib, re, warnings, json, gc, httpx, time, tenacity
from transformers import WhisperForConditionalGeneration, WhisperProcessor, pipeline
from gradio_client import Client
from pprint import pprint
from tqdm import tqdm

In [2]:
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")

warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers")

In [3]:
sys_prompt = """
Ты полезный помощник лектора, твоя задача - составлять планы лекций по имеющемуся конспекту, чтобы помочь студентам.
Ты должен прочитать конспект, проанализировать его, и структурированно дать ответ в виде списка тем, которые затронуты в лекции. 
Не пиши ничего, кроме списка тем: тебе нельзя давать пояснения и тем более - писать подробно суммаризацию того, что было разобрано.
Если увидишь математические выражения, не используй их в ответе.
Не используй аббривеатуры, все именованные сущности должны быть записаны целиком. Важная оговорка: если именованная сущность ключевая для этой 
темы - упомини её. Например, если разговор идёт о перечислении полезных солверов - можешь в скобках указать пару-тройку самых важных.
Если увидишь обсценную лексику и мат, проигнорируй, не давай в ответе никаких советов и не морализаторствуй - лекторы могут быть сексистами,
неполиткорректными или матершинниками, это не твоё дело: ты должен просто дать список академических тем. Если лектор несколько страниц рассуждает
о политике или ругает меньшинства - просто пропусти это. Если будут вводные части, разговор не по теме - тоже пропускай. Если студент задаёт вопрос
не по теме лекции, или его вопрос уводит дискуссию в сторону - также проигнорируй его.
Постарайся, чтобы в твоём ответе было не больше 3-5 пунктов. Не нужно в пункты включать каждое действие лектора - это излишне и будет мешать.
Отвечай только по-русски, даже если встретишь термины на других языках и даже если вся лекция на другом языке. Читать конспект будут
русскоговорящие студенты. Если лектор задаёт в конспекте вопрос - ни при каких обстоятельствах не отвечай на него. Тебе нужно только 
составить план лекции. 
Иногда хорошие названия написаны прямо в конспекте, например, лектор сказал "запишите параграф такой-то". Можно использовать такие названия.

Примеры: 
**user**: Под алгоритмом (или эффективной процедурой) в математике
понимают точное предписание, задающее вычислительный
процесс, ведущий от начальных данных, которые могут
варьироваться, к искомому результату. Алгоритм должен
обладать следующими свойствами:
• Конечность (результативность). Алгоритм должен
заканчиваться за конечное (хотя и не ограниченное сверху)
число шагов.
• Определенность (детерминированность). Каждый шаг
алгоритма и переход от шага к шагу должны быть точно
определены и каждое применение алгоритма к одним и тем
же исходным данным должно приводить к одинаковому
результату.
• Простота и понятность. Каждый шаг алгоритма должен быть
четко и ясно определен, чтобы выполнение алгоритма
можно было «поручить» любому исполнителю (человеку или
механическому устройству).
• Массовость. Алгоритм задает процесс вычисления для
множества исходных данных (чисел, строк букв и т.п.), он
представляет общий метод решения класса задач.
Пример. Алгоритм Евклида нахождения наибольшего общего
делителя двух целых положительных чисел a и b НОД(a, b).
Даны два целых числа a и b, найти НОД(a, b).
Выполнить следующие шаги:
1. Если a < b, то поменять их местами.
2. Разделить нацело a на b; получить остаток r.
3. Если r = 0, то НОД(a, b) = b.
4. Если r 6= 0, заменить: a на b, b на r и вернуться к шагу 2.
Не имея такого определения, невозможно доказать, что задача
алгоритмически неразрешима, т.е. алгоритм ее решения никогда
не удастся построить.
Тезис Тьюринга–Чёрча. Для любой интуитивно вычислимой
функции существует вычисляющая её значения машина
Тьюринга.
Тезис Тьюринга–Чёрча невозможно строго доказать или
опровергнуть, так как он устанавливает эквивалентность между
строго формализованным понятием частично вычислимой
функции и неформальным понятием вычислимости.
Алфавит — это конечное множество Ap элементов ai
:
Ap = {a1
, a2, . . . , ap}.
Элементы алфавита Ap называются символами.
Последовательность из m символов алфавита Ap называется
словом длины m над алфавитом Ap: ai1
ai2
. . . aim
Слово длины 0 называется пустым словом и обозначается ε.
Множество всех слов над алфавитом Ap:
A
∗
p = {ε} ∪ Ap ∪ A
2
p ∪ . . . ∪ A
m
p ∪ . . . =
[∞
m=0
A
m
p
.
Длину слова w ∈ A
∗
p будем обозначать |w|,
в частности, для пустого слова |ε| = 0.
Утверждение. Для любой пары алфавитов A и B можно
выполнить кодирование алфавита A с помощью алфавита B и
обратно, возможно, с применением дополнительно служебного
символа ı («конец кода символа»).
Следствие. Кодирование позволяет ограничиться одним
алфавитом.
Обычно рассматриваются A1 или A2
Задача обработки информации — это задача построения
частичного отображения (функции) F : A
∗ → A
∗
.
Утверждение. Существует взаимно-однозначное отображение
# : A
∗ ↔ N0, где N0 — множество целых неотрицательных чисел,
которое любому слову w ∈ A
∗
ставит в соответствие его номер
n ∈ N0. (Это отображение # и называется нумерацией.)
A
∗ A
∗
N0 N0
#
F
f
#−1
Таким образом:
1. каждый алгоритм F : A
∗ → A
∗ определяет частично
вычислимую функцию f : N0 → N0;
2. каждая частично вычислимая функция f : N0 → N0
определяет алгоритм F : A
∗ → A
∗
.

Машина-автомат: предъявляется любое исходное слово w ∈ A
∗
,
а в результате обработки получается слово v = F(w).
Каждая частичная функция F, для которой можно построить МТ,
называется вычислимой по Тьюрингу
Алфавит состояний Q = {q0, q1
, q2, . . . , qn}
Рабочий алфавит S = A ∪ A
0
A — алфавит входных символов
A
0
— алфавит вспомогательных символов (маркеров)
Лента, размеченная на ячейки (пустая ячейка — Λ)
Управляющая головка (УГ)
Рабочая ячейка (РЯ)
Начальное состояние q0, состояние останова qs
Начальные данные — слова из A
∗
Конфигурация МТ: hn, F, qi, где n — номер текущей рабочей ячейки,
F : Z → S — текущая запись на ленте, q — текущее состояние.
Позиция МТ: пара hn, qi.
Такт работы МТ:
hсостояние, символi → hсостояние, символ, направлениеi
 
**assistant**: 1. Неформальное (интуитивное) определение алгоритма 
        2. Почему необходимо формальное определение алгоритма
        3. Формализация понятия алгоритма.
        4. Машина Тьюринга (МТ).

**user**: В комбинаторике существуют принципы для решения различных задач.
Принципы
1) Принцип сложения
Если у нас имеются два непересекающихся множества, то число элементов в
объединении равно сумме чисел элементов в этих множествах:
|A| ` |B| “ |A Y B|, A X B “ ∅, (1)
где |A| - число элементов в множестве или мощность множества.
2) Принцип умножения
Рассмотрим декартово произведение A ˆ B “ tpa, bq|a P A, b P Bu. Мощность
этого множества равна произведению мощностей A и B:
|A ˆ B| “ |A| ˆ |B|. (2)
3) Принцип взаимно однозначного соответствия
A Ø B, т.е. каждому элементу одного множества сопоставлен единственный
элемент другого множества ñ |A| “ |B|.
4) Принцип двойного подсчета
Число элементов в множестве можно посчитать двумя разными способами.
Если оба способа приводят к правильному ответу, то получаем равенство двух
выражений. Рассмотрим пример задачи, где ничего считать не надо, однако
используется этот принцип.
Пример (задача о паркете) Пусть у нас имеется прямоугольная комната, в которой положены прямоугольные паркетинки разной формы. Известно,
что одно из измерений паркетинки (длина или ширина) равно целому числу.
Тогда, если комнату можно замостить паркетинками, то у этой комнаты одно из измерений тоже будет целым числом. Для решения задачи применим
принцип двойного подсчета.
а) Введем ДПСК, где точка начала координат лежит в вершине комнаты.
Рассмотрим всевозможные целые точки (обе координаты точки целые),
которые являются вершинами паркетинок. Для каждой паркетинки посчитаем число вершин, которые являются целыми точками. Получится,
что каждая паркетинка содержит 0/2/4 целые точки. Просуммировав по
всем паркетинкам число целых точек получим четное число.

б) Возьмем произвольную целую точку, являющуюся вершиной одной из
паркетинок, и посчитаем сколько раз она будет участвовать в сумме, т.е.
для какого количества паркетинок эта вершина является общей. Тогда,
если целая точка не является вершиной комнаты, она принадлежит 2 или
4 паркетинкам (и сумма по всем таким точкам будет четной). Выходит,
что и сумма целых точек по вершинам комнаты должна быть четной
(так как сумма, полученная в пункте а) была четной). Но у нас заведомо
есть вершина комнаты, являющаяся целой точкой - это начало координат
(0, 0). Значит, еще как минимум одна вершина комнаты является целой.
Легко видеть, что тогда как минимум одна сторона комнаты будет целой,
что и требовалось доказать.
Обобщение принципов
1) Обобщение принципа сложения
Если у нас имеются n попарно непересекающихся множеств, то мощность объединения множеств равна сумме мощностей множеств:
|A1 Y ¨ ¨ ¨ Y An| “ ÿ
|Ai
|, Ai X Aj “ ∅, i ‰ j. (3)
2) Обобщение принципа умножения
Рассмотрим декартово произведение n множеств A ˆ B “ tpa1, . . . , anq|ai P
Ai
, i “ 1, nu. Мощность этого множества равна произведению мощностей:
|A1 ˆ ¨ ¨ ¨ ˆ An| “ |A1| ˆ ¨ ¨ ¨ ˆ |An|. (4)
Формула включения и исключения
Правило сложения в случае пересечения множеств превращается в формулу включения и исключения
|A Y B| “ |A| ` |B| ´ |A X B| (5)
|A1 Y ¨ ¨ ¨ Y An| “ ÿn
i“1
|Ai
| ´ ÿ
1ďiďjďn
|Ai X Aj
| ` ÿ
1ďiďjďkďn
|Ai X Aj X Ak| ´ ¨ ¨ ¨ `
` p´1q
k`1 ÿ
1ďi1ď¨¨¨ďikďn
|Ai1 X ¨ ¨ ¨ X Aik
| ` p´1q
n`1
|A1 X ¨ ¨ ¨ X An| (6)
Для упрощения записи можно рассмотреть k - элементное множество I, состоящее
из индексов. Тогда получим следующую формулу
ÿn
k“1
p´1q
k`1 ÿ
|I|“k
|
č
iPI
Ai
|.
Задача Посчитаем количество элементов в множестве A “ t1 ď k ď n,pk.nq “ 1u.Решение Мощность будет равняться значению функции Эйлера в точке n.
|A| “ φpnq “ pp
α1
1 ´ p
α1´1
1
q ´ pp
αn
n ´ p
αn´1
n
q “ n
ˆ
1 ´
1
p1
˙
. . . ˆ
1 ´
1
pn
˙
Определение Произвольная перестановка - последовательность чисел от 1 до n,
переставленных в каком-то порядке.
π “ πp1q. . . πpnq, πpiq P t1, . . . , nu, πpiq ‰ πpjq, i ‰ j. Перестановка π P S1, |S1| “ n!.
Определение Неподвижная точка перестановки πpiq “ i.
Проиллюстрируем формулу включений и исключений следующей задачей и теоремой.
Задача о числе перестановок без неподвижных точек Сколько перестановок не имеет неподвижных точек? Эта задача будет подробнее рассмотрена на
семинарах.
Пусть µ - аддитивная мера множества, тогда
µpA Y Bq “ µpAq ` µpBq ´ µpA X Bq
Теорема Лапласа Рассмотрим множество
A “ tpx1, . . . , xnq, k ´ 1 ď
ÿn
i“1
xi ď nu X r0, 1s
n
.
Какова вероятность того, сумма координат случайной точки из n-мерного куба
будет лежать в таких пределах?
PpAq “ 1
n!
ÿ
k
i“0
p´1q
i
pk ´ iq
n
ˆ
n ` 1
i
˙
Размещение шаров по ящикам
Существует n ящиков и m коробок. Общее число размещений шаров по ящикам
равно mn
.
Если можно помещать не больше 1 шара в ящик, то общее число размещений
равно mpm ´ 1q. . .pm ´ n ` 1q “ rmsn - убывающий субфакториал. Возрастающий
субфакториал rms
n “ mpm ` 1q. . .pm ` n ´ 1q.
Если по условию задачи каждый ящик не должен быть пустым, то следует рассмотреть инъективное отображение f : t1, . . . , nu Ñ t1, . . . , mu.
Существует алфавит A “ ta1, . . . , anu. Произвольное слово ai1
, . . . , aim. Am - множество всех слов длины m.
Задача Найти число всех слов длины m в данном алфавите.
Решение |Am| “ |A|
m.
Задача Найти число слов заданной длины с заданным распределением букв mi
.
Решение Am
m1,...,mn
- число всех слов в алфавите с заданным распределением
букв. Это задача совпадает с задачей о числе перестановок с повторениями, поэтому
получаем m!
m1!...mn!
.Полиномиальная теорема
px1 ` ¨ ¨ ¨ ` xnq
m “
ÿ
m1`¨¨¨`mn“m,miě0
x
m1
1
. . . xmn
n
m!
m1! . . . mn!
При n “ 2 получаем биномиальную теорему.
**assistant**: 
1. Принципы 
2. Обобщение принципов
3. Формула включения и исключения
4. Размещение шаров по ящикам

Лекция для составления плана:
"""

In [4]:
BATCH_SIZE = 5
NUM_ATTEMPTS = 3
MIN_SYMBOLES = 500
BATCH_SIZE_ASR = 12
TEMPERATURE = 0.5
MAX_TOKENS = 1024
MAX_DURATION_SECONDS = 2 * 60 * 60
SPLITTING_LEN = 400

folder_path = pathlib.Path("./data_raw/")
lections_names = list(folder_path.glob("*.mp3"))
len_of_files = len(lections_names)

torch_dtype = torch.float16
np_dtype = "float16"
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
device = torch.device(device)

whisper = WhisperForConditionalGeneration.from_pretrained(
    "antony66/whisper-large-v3-russian", torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)

asr_processor = WhisperProcessor.from_pretrained("antony66/whisper-large-v3-russian")

asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=whisper,
    tokenizer=asr_processor.tokenizer,
    feature_extractor=asr_processor.feature_extractor,
    max_new_tokens=256,
    chunk_length_s=30,
    batch_size=BATCH_SIZE_ASR,
    return_timestamps=False,
    torch_dtype=torch_dtype,
    device=device,
)

final_dataset = []

Device set to use cuda


In [5]:
@tenacity.retry(
    retry=tenacity.retry_if_exception_type(httpx.ConnectTimeout),
    stop=tenacity.stop_after_attempt(5),
    wait=tenacity.wait_exponential(multiplier=1, max=60),
    reraise=True
)
def create_client_with_retry():
    return Client("ritzy88/MyNewChatApp")
    
@tenacity.retry(
    retry=tenacity.retry_if_exception_type(httpx.HTTPError),
    stop=tenacity.stop_after_attempt(5),
    wait=tenacity.wait_exponential(multiplier=1, max=60),
    reraise=True
)
def generate_with_retry(client : Client, conspect : str, system_message : str):
    return client.predict(
                    message=conspect,
                    system_message=sys_prompt,
                    max_tokens=MAX_TOKENS,
                    temperature=TEMPERATURE,
                    top_p=0.95,
                    api_name="/chat",
                )

In [ ]:
for i in range(0, len_of_files, BATCH_SIZE):
    batch_files = lections_names[i:i + BATCH_SIZE]
    print(f"\n--- Батч {i // BATCH_SIZE + 1} / { (len_of_files - 1) // BATCH_SIZE + 1 } ---")

    audio_arrays = []
    conspects = []
    plans = []

    # Этап 1: Загрузка и транскрибация аудио
    for lection_mp3 in batch_files:
        try:
            y, sr = librosa.load(lection_mp3, sr=16000)
            y = y.astype(np_dtype)
            duration_seconds = len(y) / sr

            if duration_seconds > MAX_DURATION_SECONDS:
                print(f"Пропущен файл (слишком длинный > 2ч): {lection_mp3.name}")
                continue

            print(f"Загружен: {lection_mp3.name}, форма: {y.shape}")

            audio_arrays.append({'raw': y, 'sampling_rate': sr})
        except Exception as e:
            print(f"Ошибка загрузки {lection_mp3.name}: {e}")
            continue

    for audio in tqdm(audio_arrays, desc="Транскрибация"):
        try:

            result = asr_pipeline(audio, generate_kwargs={"task" : 'transcribe', "language" : 'russian'})["text"].strip()

            result = re.sub(r'(\b\w+\b)(?:\s+\1){2,}', r'\1', result)
            result = re.sub(r'\s{2,}', '', result)
            
            # Проверка: не пустой ли текст
            if not result or len(result) < MIN_SYMBOLES:
                print("Пустая или слишком короткая транскрипция — пропущено")
                continue

            # Делим на 3 части
            len_symb = len(result)
            parts = [
                result[:len_symb // 3 + SPLITTING_LEN],
                result[len_symb // 3 - SPLITTING_LEN: 2 * len_symb // 3 + SPLITTING_LEN],
                result[2 * len_symb // 3 - SPLITTING_LEN:]
            ]
            conspects.extend([p.strip() for p in parts if len(p.strip()) > 5])
          #  conspects.append(result)

        except Exception as e:
            print(f"Ошибка ASR: {e}")
            continue

    # Этап 2: Генерация планов через LLM
    try:
        client = create_client_with_retry()
    except httpx.ConnectTimeout:
        print("Не удалось подключиться к API после нескольких попыток.")

    for conspect in tqdm(conspects, desc="Генерация планов"):
        for attempt in range(NUM_ATTEMPTS):
            try:
                result = generate_with_retry(
                    client,
                    conspect,
                    sys_prompt
                )
                plans.append(result.strip())
                break
            except httpx.HTTPError:
                print("Не удалось воспользоваться API после нескольких попыток.")
            except Exception as e:
                print(f"Ошибка LLM (попытка {attempt + 1}): {e}")
                time.sleep(2 ** attempt)
        else:
            print("Все попытки исчерпаны — пропущено")
            plans.append("")

    # Этап 3: Сборка батча и добавление в общий датасет
    for text, plan in zip(conspects, plans):
        if not text or not plan.strip(): 
            continue
        entry = {
            "id": len(final_dataset) + 1,
            "text": text,
            "plan": plan
        }
        final_dataset.append(entry)

    # Опционально: сохранение чекпоинта
    with open("dataset_checkpoint.json", "w", encoding="utf-8") as f:
        json.dump(final_dataset, f, ensure_ascii=False, indent=4)

    del audio_arrays, conspects, plans
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Батч {i // BATCH_SIZE + 1} обработан.")


with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, ensure_ascii=False, indent=4)

print(f"Датасет сохранён: {len(final_dataset)} записей.")


--- Батч 1 / 354 ---
Загружен: 0004_CQQ-dzrMHog_12..mp3, форма: (76345686,)
Загружен: 0005_-yGsuJtXUao_11..mp3, форма: (68763989,)
Загружен: 0006_FOd4CO2S-ic_3..mp3, форма: (85851819,)
Загружен: 0007_z3tWszHoV1s_2..mp3, форма: (79755264,)
Загружен: 0008_Ac2hkVT4XUc_3..mp3, форма: (75392000,)


Транскрибация: 100%|██████████| 5/5 [08:37<00:00, 103.56s/it]


Loaded as API: https://ritzy88-mynewchatapp.hf.space ✔


Генерация планов: 100%|██████████| 15/15 [01:26<00:00,  5.80s/it]


Батч 1 обработан.

--- Батч 2 / 354 ---
Загружен: 0008_njA5lNqftZQ_._._-_No15_4.mp3, форма: (76179115,)
Загружен: 0009_-zEyIcV1M6Q_4..mp3, форма: (72660651,)
Загружен: 0009_uSl59DMmU3Q_._._-_6_No21_19.05.2025.mp3, форма: (86782976,)
Загружен: 0010_VHRWnnlAETs_._._15_4.mp3, форма: (85950123,)
Загружен: 0010_ZXL66FGI2hw_4..mp3, форма: (68731563,)


...

Транскрибация: 100%|██████████| 5/5 [09:56<00:00, 119.33s/it]


Loaded as API: https://ritzy88-mynewchatapp.hf.space ✔


Генерация планов: 100%|██████████| 15/15 [01:28<00:00,  5.91s/it]


Батч 10 обработан.

--- Батч 11 / 354 ---
Загружен: 0027_wkDc_VxpgRk_._._-_No4_2.mp3, форма: (87675205,)
Загружен: 0028_3csXzNNBt1E_1..mp3, форма: (82795520,)
Загружен: 0028_dlS2FwoCiZI_._16._._._..mp3, форма: (88925184,)
Загружен: 0028_v9Dpt1FP4N8_._._-_No3_2.mp3, форма: (86083247,)
Загружен: 0029_GB5h-CXx39k_N_..mp3, форма: (90150571,)


Транскрибация:  60%|██████    | 3/5 [06:18<04:08, 124.21s/it]